In [1]:
from __future__ import annotations
import json
import re
from pathlib import Path
from typing import Any, Dict, List
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

TOOLS_PATH = Path("/scratch4/home/akrik/NTILC/data/ToolCall15/tools.json")
MODEL_NAME = "Qwen/Qwen3.5-27B"
DEVICE = "cuda:2"
DTYPE = "auto"

TOOL_INDEX = 0
REQUEST_COUNT = 8
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.8
TOP_P = 0.95

/scratch4/home/akrik/base/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SYSTEM_PROMPT = """You create short, realistic user requests for tool-routing datasets.
Each string must be a natural user request.
Do not mention tool names, JSON, schemas, or implementation details.
Do not number the items."""

In [3]:
def load_tools(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    tools = payload.get("tools")
    if not isinstance(tools, list):
        raise ValueError(f"Expected 'tools' to be a list in {path}")
    return tools


def format_parameters(tool: Dict[str, Any]) -> str:
    parameters = tool.get("parameters", {})
    properties = parameters.get("properties", {})
    required = set(parameters.get("required", []))
    if not properties:
        return "- no parameters"

    lines: List[str] = []
    for name, spec in properties.items():
        if not isinstance(spec, dict):
            continue
        parts = [f"- {name}: {spec.get('type', 'any')}"]
        if spec.get("enum"):
            parts.append(f"choices={spec['enum']}")
        if "default" in spec:
            parts.append(f"default={spec['default']}")
        if name in required:
            parts.append("required")
        lines.append(", ".join(parts))
    return "\n".join(lines) if lines else "- no parameters"


def build_prompt(tool: Dict[str, Any], count: int) -> str:
    name = tool.get("name", "").strip()
    description = tool.get("description", "").strip()
    parameter_text = format_parameters(tool)
    return f"""Create {count} different user requests for this tool.

    Tool name: {name}
    Tool description: {description}
    Parameters:
    {parameter_text}

    Requirements:
    - The request should clearly map to this tool.
    - Keep the language simple and direct.
    - Vary names, locations, dates, numbers, and phrasing.
    - Some requests can mention optional parameters when relevant.
    - Avoid duplicates.
    - Each user request should start with <request> and end with </request>
    - Only output the requests"""


def normalize_query(text: str) -> str:
    text = text.strip().strip('"').strip("'")
    text = re.sub(r"\s+", " ", text)
    return text


def unique_preserve_order(items: List[str]) -> List[str]:
    seen = set()
    result: List[str] = []
    for item in items:
        key = item.casefold()
        if item and key not in seen:
            seen.add(key)
            result.append(item)
    return result


def parse_generated_queries(raw_text: str) -> List[str]:
    parts = raw_text.split("</think>")
    post_think_text = parts[-1]
    matches = re.findall(r"<request>(.*?)</request>", post_think_text, flags=re.DOTALL)
    queries = [normalize_query(match) for match in matches if normalize_query(match)]
    return unique_preserve_order(queries)


def resolve_dtype(dtype_name: str, device: str) -> torch.dtype:
    if dtype_name == "float16":
        return torch.float16
    if dtype_name == "bfloat16":
        return torch.bfloat16
    if dtype_name == "float32":
        return torch.float32
    if device.startswith("cpu"):
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_available() else torch.float32


def load_generator(model_name: str, device: str, dtype: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs: Dict[str, Any] = {
        "trust_remote_code": True,
        "torch_dtype": resolve_dtype(dtype, device),
    }
    if device == "auto":
        model_kwargs["device_map"] = "auto"

    model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if device != "auto":
        model = model.to(device)
    model.eval()
    return tokenizer, model


@torch.inference_mode()
def generate_queries(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
    device: str,
) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]

    if hasattr(tokenizer, "apply_chat_template"):
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    else:
        text = f"{SYSTEM_PROMPT}\n\n{prompt}"

    encoded = tokenizer(text, return_tensors="pt")
    if device != "auto":
        encoded = {key: value.to(device) for key, value in encoded.items()}

    generated = model.generate(
        **encoded,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    prompt_tokens = encoded["input_ids"].shape[1]
    new_tokens = generated[0][prompt_tokens:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


In [4]:
tools = load_tools(TOOLS_PATH)
tool = tools[TOOL_INDEX]

print("Selected tool:")
print(json.dumps(tool, indent=2))

prompt = build_prompt(tool, REQUEST_COUNT)
print("\nPrompt:\n")
print(prompt)

Selected tool:
{
  "name": "web_search",
  "description": "Search the web for current information",
  "parameters": {
    "type": "object",
    "properties": {
      "query": {
        "type": "string"
      },
      "max_results": {
        "type": "integer",
        "default": 5
      }
    },
    "required": [
      "query"
    ]
  }
}

Prompt:

Create 8 different user requests for this tool.

    Tool name: web_search
    Tool description: Search the web for current information
    Parameters:
    - query: string, required
- max_results: integer, default=5

    Requirements:
    - The request should clearly map to this tool.
    - Keep the language simple and direct.
    - Vary names, locations, dates, numbers, and phrasing.
    - Some requests can mention optional parameters when relevant.
    - Avoid duplicates.
    - Each user request should start with <request> and end with </request>
    - Only output the requests


In [5]:
tokenizer, model = load_generator(MODEL_NAME, DEVICE, DTYPE)

raw_output = generate_queries(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    device=DEVICE,
)

parsed_queries = parse_generated_queries(raw_output)

print("Raw output:\n")
print(raw_output)
print("\nParsed queries:\n")
for i, query in enumerate(parsed_queries, start=1):
    print(f"{i}. {query}")

`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 97336.17it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 851/851 [00:00<00:00, 1398.04it/s, Materializing param=model.norm.weight]                              


Raw output:

Thinking Process:

1.  **Analyze the Request:**
    *   Task: Create 8 different user requests for a specific tool.
    *   Tool Name: `web_search`
    *   Tool Description: Search the web for current information
    *   Parameters: `query` (string, required), `max_results` (integer, default=5)
    *   Constraints:
        *   Natural user requests (no tool names, JSON, schemas, etc.).
        *   Simple and direct language.
        *   Vary names, locations, dates, numbers, phrasing.
        *   Some requests can mention optional parameters (like number of results) naturally.
        *   Avoid duplicates.
        *   Format: Each request must start with `<request>` and end with `</request>`.
        *   Output: Only the requests (no numbering, no extra text).

2.  **Understand the Tool:**
    *   `web_search` is for finding current information on the internet.
    *   Users will ask questions or state topics they want to know about.
    *   `max_results` can be hinted at 